# Multilayer Perceptron Classification of Plant Alliances

This notebook trains a Multilayer Perceptron (MLP) classifier to predict phytosociological alliances from vegetation relevé vectors using Optuna for hyperparameter optimization.

## 1. Load Dependencies

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, log_loss
from scikeras.wrappers import KerasClassifier

## 2. Load and Prepare Data

In [ ]:
# Load vectorized relev? data
file_path = 'matriu_dades/output_pirineus_3_corrected.txt'
with open(file_path, 'r') as file:
    loaded_vector_dic = json.load(file)

# Build feature matrix
df = pd.DataFrame.from_dict(loaded_vector_dic, orient='index', columns=['List', 'Label'])
df = pd.DataFrame(df['List'].tolist(), index=df.index)
df.columns = [f'Column_{i+1}' for i in range(len(df.columns))]

# Add target labels
labels = [loaded_vector_dic[idx][1] for idx in df.index]
df['Target'] = labels

# Filter classes with fewer than 20 samples
MIN_SAMPLES = 20
label_counts = df['Target'].value_counts()
valid_labels = label_counts[label_counts >= MIN_SAMPLES].index.tolist()
df_filtered = df[df['Target'].isin(valid_labels)]

print(f"Dataset shape: {df_filtered.shape}")
print(f"Number of classes: {len(valid_labels)}")

## 3. Encode Labels and Split Data

In [ ]:
label_encoder = LabelEncoder()

X = df_filtered.drop(columns='Target')
y = label_encoder.fit_transform(df_filtered['Target'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=8
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Autoencoder for Dimensionality Reduction

Train a sparse autoencoder to reduce the high-dimensional input to a compressed representation.

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from sklearn.metrics import mean_squared_error

np.random.seed(42)
tf.random.set_seed(42)

# Normalize data (values range from 0 to 9)
X_norm = X / 9

input_dim = X_norm.shape[1]
encoding_dim = 500

# Build autoencoder
input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.fit(X_norm, X_norm, epochs=40, batch_size=256, shuffle=True, validation_split=0.2)

# Transform data
X_reduced = encoder.predict(X_norm)

# Evaluate reconstruction error
X_reconstructed = autoencoder.predict(X_norm)
mse = mean_squared_error(X_norm, X_reconstructed)
print(f"Reconstruction MSE: {mse:.6f}")
print(f"Reduced dimensions: {X_reduced.shape[1]}")

## 5. Save Autoencoder

In [ ]:
encoder.save('models/encoder.h5')
autoencoder.save('models/autoencoder.h5')
print("Autoencoder models saved")

## 6. Prepare Reduced Data for Training

In [ ]:
# Normalize the reduced representations
X_normalized = (X_reduced - np.min(X_reduced)) / (np.max(X_reduced) - np.min(X_reduced))

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_normalized, y, test_size=0.3, stratify=y, random_state=8
)

print(f"Reduced train shape: {X_train_r.shape}")
print(f"Reduced test shape: {X_test_r.shape}")

## 7. Hyperparameter Optimization with Optuna

In [ ]:
num_classes = len(np.unique(y))
all_classes = np.unique(y)

def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    n_units = trial.suggest_int('n_units', 32, 256)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'sigmoid'])
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-1)
    epochs = trial.suggest_int('epochs', 10, 50)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
    fold_losses = []

    for train_index, val_index in skf.split(X_normalized, y):
        X_train_cv, X_val_cv = X_normalized[train_index], X_normalized[val_index]
        y_train_cv, y_val_cv = y[train_index], y[val_index]

        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Input(shape=(X_train_cv.shape[1],)))
        for _ in range(n_layers):
            model.add(tf.keras.layers.Dense(n_units, activation=activation))
        model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

        if optimizer_name == 'adam':
            optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        elif optimizer_name == 'sgd':
            optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
        else:
            optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)

        model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        model.fit(X_train_cv, y_train_cv, epochs=epochs,
                  validation_data=(X_val_cv, y_val_cv), verbose=0)

        val_predictions = model.predict(X_val_cv)
        fold_loss = log_loss(y_val_cv, val_predictions, labels=all_classes)
        fold_losses.append(fold_loss)

    return np.mean(fold_losses)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, n_jobs=10)
best_params = study.best_params
print(f"Best parameters: {best_params}")

## 8. Build and Train Best Model

In [ ]:
def create_model(params):
    """Create MLP model from hyperparameter dictionary."""
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(X_train_r.shape[1],)))
    for _ in range(params['n_layers']):
        model.add(tf.keras.layers.Dense(params['n_units'], activation=params['activation']))
    model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

    if params['optimizer'] == 'adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])
    elif params['optimizer'] == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=params['learning_rate'])
    else:
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=params['learning_rate'])

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

best_model = create_model(best_params)
history = best_model.fit(
    X_train_r, y_train_r,
    epochs=best_params['epochs'],
    validation_data=(X_test_r, y_test_r),
    verbose=1
)

test_loss, test_accuracy = best_model.evaluate(X_test_r, y_test_r)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

## 9. Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training and Validation Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.set_title('Training and Validation Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 10. Cross-Validation

In [ ]:
model_cv = KerasClassifier(model=create_model, params=best_params, epochs=best_params['epochs'], verbose=0)
skf = StratifiedKFold(n_splits=5)
cv_scores = cross_val_score(model_cv, X_normalized, y, cv=skf, scoring='accuracy')

print(f"Cross-validated accuracy per fold: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 11. Classification Report

In [ ]:
y_pred_prob = best_model.predict(X_test_r)
y_pred = np.argmax(y_pred_prob, axis=1)

class_names = label_encoder.classes_
report = classification_report(
    label_encoder.inverse_transform(y_test_r),
    label_encoder.inverse_transform(y_pred),
    target_names=class_names, zero_division=0, output_dict=True
)
report_df = pd.DataFrame(report).transpose()
print(f"Test Accuracy: {accuracy_score(y_test_r, y_pred):.4f}")
report_df.to_excel('cls_report_test_mlp.xlsx')
report_df.head(10)

## 12. Detailed Predictions with Top-3 Probabilities

In [ ]:
def get_top_3_predictions(proba_vector, label_mapping):
    """Get top 3 predicted classes and their probabilities."""
    top_3_indices = np.argsort(proba_vector)[-3:][::-1]
    top_3_proba = proba_vector[top_3_indices]
    top_3_labels = [label_mapping[idx] for idx in top_3_indices]
    return top_3_labels, top_3_proba

# Recover original relev? IDs using the same random state
X_train_a, X_test_a, _, _ = train_test_split(X, y, test_size=0.3, stratify=y, random_state=8)
label_mapping = dict(enumerate(label_encoder.classes_))

# Generate predictions
prob_test = best_model.predict(X_test_r)
top_3_test = [get_top_3_predictions(p, label_mapping) for p in prob_test]

df_results = pd.DataFrame({
    'releve_id': X_test_a.index,
    'prediction': label_encoder.inverse_transform(np.argmax(prob_test, axis=1)),
    'true_label': label_encoder.inverse_transform(y_test_r),
    'top_3_labels': [t[0] for t in top_3_test],
    'top_3_probabilities': [t[1] for t in top_3_test]
})

df_results.to_excel('predictions_test_mlp.xlsx', index=False)
df_results.head()

## 13. Train Final Model and Save

In [ ]:
# Train on full dataset for deployment
final_model = create_model(best_params)
final_model.fit(X_normalized, y, epochs=best_params['epochs'], verbose=1)

final_model.save('models/mlp_final_model.h5')
print("Final model saved to models/mlp_final_model.h5")